In [40]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt # for making figures
%matplotlib inline

In [41]:
words = open('names.txt', 'r').read().splitlines()

In [42]:
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}

In [43]:
# build the dataset

block_size = 3 # context length: how many characters do we take as input to predict the next one?
X, Y = [], [] # input, target set
for w in words[:5]:
  #print(w)
  context = [0] * block_size
  for ch in w + '.':
    ix = stoi[ch]
    X.append(context)
    Y.append(ix)
    #print(''.join(itos[i] for i in context), '--->', itos[ix])
    context = context[1:] + [ix] # crop and append
  
X = torch.tensor(X)
Y = torch.tensor(Y)

In [44]:
C = torch.randn((27, 2)) # Embeddings table

In [45]:
emb = C[X]
emb.shape

torch.Size([32, 3, 2])

In [46]:
# To form the hidden layer, we need to add weights and biases to 32,3,2 

In [47]:
W1 = torch.randn((6, 100)) # 3 inputs with 2 embedding each, 6, let there be 100 neurons ahead
b1 = torch.randn(100) # biases for each neuron

In [48]:
# turning 32,3,2 to 32,6
#1.
torch.cat([emb[:, 0, :], emb[:, 1, :], emb[:, 2, :]], 1).shape

torch.Size([32, 6])

In [49]:
# 2.
torch.cat(torch.unbind(emb,1 ), 1).shape

torch.Size([32, 6])

In [50]:
# 3. Most efficient and simplest
emb.view(32,6)

tensor([[-0.3957,  0.2210, -0.3957,  0.2210, -0.3957,  0.2210],
        [-0.3957,  0.2210, -0.3957,  0.2210,  1.3927,  1.0425],
        [-0.3957,  0.2210,  1.3927,  1.0425, -1.4891,  0.0775],
        [ 1.3927,  1.0425, -1.4891,  0.0775, -1.4891,  0.0775],
        [-1.4891,  0.0775, -1.4891,  0.0775,  0.7974, -0.0577],
        [-0.3957,  0.2210, -0.3957,  0.2210, -0.3957,  0.2210],
        [-0.3957,  0.2210, -0.3957,  0.2210,  1.7241,  1.8716],
        [-0.3957,  0.2210,  1.7241,  1.8716,  0.4841, -0.0799],
        [ 1.7241,  1.8716,  0.4841, -0.0799,  0.7574, -0.3950],
        [ 0.4841, -0.0799,  0.7574, -0.3950, -0.7029, -1.1893],
        [ 0.7574, -0.3950, -0.7029, -1.1893,  0.7574, -0.3950],
        [-0.7029, -1.1893,  0.7574, -0.3950,  0.7974, -0.0577],
        [-0.3957,  0.2210, -0.3957,  0.2210, -0.3957,  0.2210],
        [-0.3957,  0.2210, -0.3957,  0.2210,  0.7974, -0.0577],
        [-0.3957,  0.2210,  0.7974, -0.0577, -0.7029, -1.1893],
        [ 0.7974, -0.0577, -0.7029, -1.1

In [51]:
# Creating the hidden layer of neurons
h = torch.tanh(emb.view(-1, 6) @ W1 + b1)

In [52]:
h.shape

torch.Size([32, 100])

In [53]:
# The final softmax layer, taking in the out of the neurons and giving it out in the form of prob
W2 = torch.randn((100, 27))
b2 = torch.randn(27)

In [54]:
logits = h @ W2 + b2

In [55]:
logits.shape

torch.Size([32, 27])

In [56]:
counts = logits.exp()

In [57]:
prob = counts / counts.sum(1, keepdims=True)

In [58]:
prob.shape

torch.Size([32, 27])

In [59]:
loss = -prob[torch.arange(32), Y].log().mean()
loss

tensor(20.2099)

In [60]:
# Making a neater form of the same

In [61]:
g = torch.Generator().manual_seed(2147483647) # for reproducibility
C = torch.randn((27, 2), generator=g)
W1 = torch.randn((6, 100), generator=g)
b1 = torch.randn(100, generator=g)
W2 = torch.randn((100, 27), generator=g)
b2 = torch.randn(27, generator=g)
parameters = [C, W1, b1, W2, b2]

In [62]:
sum(p.nelement() for p in parameters) # number of parameters in total

3481

In [63]:
for p in parameters:
  p.requires_grad = True

In [68]:
for i in range(10):
    #Forward pass
    emb = C[X] 
    h = torch.tanh(emb.view(-1, 6) @ W1 + b1)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, Y) # Using the entropy function to find loss since it is more effiecient for fwd and bcwd pass

    #Backward pass
    for p in parameters:
        p.grad = None
    loss.backward()

    #Update
    for p in parameters:
        p.data += -0.1 * p.data
print(loss.item())

3.2952988147735596


In [69]:
# Testing the model on a portion of the dataset now

In [70]:
lre = torch.linspace(-3, 0, 1000)
lrs = 10**lre

In [84]:
lri = []
lossi = []

for i in range(1000):
  
  # minibatch construct
  ix = torch.randint(0, X.shape[0], (32,))
  
  # forward pass
  emb = C[X[ix]] # (32, 3, 2)
  h = torch.tanh(emb.view(-1, 6) @ W1 + b1) # (32, 100)
  logits = h @ W2 + b2 # (32, 27)
  loss = F.cross_entropy(logits, Y[ix])
  #print(loss.item())
  
  # backward pass
  for p in parameters:
    p.grad = None
  loss.backward()
  
  # update
  #lr = lrs[i]
  lr = 0.01
  for p in parameters:
    p.data += -lr * p.grad

  # track stats on whats the best update rate
  lri.append(lre[i])
  lossi.append(loss.log10().item())

#print(loss.item())

In [85]:
emb = C[X[ix]] # (32, 3, 2)
h = torch.tanh(emb.view(-1, 6) @ W1 + b1) # (32, 100)
logits = h @ W2 + b2 # (32, 27)
loss = F.cross_entropy(logits, Y[ix])
loss

tensor(2.5931, grad_fn=<NllLossBackward0>)